In [ ]:

# Fig 2a, 3a, 4a: 2d to 1d decode examples

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import os
import pandas as pd
import numpy as np

In [ ]:
from paths import DATA_DIR, fig_dir

# common schema
from spyglass.common import Session, TrackGraph

# custom schema
from alison_decoding import ClusterlessAcausalResultsSummary, ClusterlessAlgorithmParameters, customize_decode_parameters
from fig_helpers import set_figure_defaults
from find_my_data import spatial_bandit_query_by_rat
from plot_decode_1d_to_2d_exs import load_decode_ex_data, plot_event_v3_1_mua_ahbeh_speed

### Load data

In [ ]:
# Params for grabbing relevant neural and behav parsed data across all animals
behavior_model_params_name = 'default_hmm_0623' #'hmm_test' #'default_hmm'
position_info_param_name='default_decoding'
remove_hpd_timepoints = True
hpd_percent = 50
hpd_threshold = 50
require_nonlocal_by_segment = False
remove_low_speed_timepoints = True
head_speed_threshold = 10

out_path = f'{DATA_DIR}/big_df_pkls/'
today_now = '20240212'
subject_ids = ['j16'] #['senor', 'chimi', 'j16', 'wilbur', 'peanut']

# Get detailed data per rat - this is more than is needed for just an ex fig but is how I did it
big_dfs = {}
for subject_id in subject_ids:
    try:
        big_dfs[subject_id] = pd.read_pickle(out_path+subject_id.lower()+'_big_df_RL_deltaq_stable'+today_now+'.pkl')
    except Exception as e:
        print('exception',e)
        
stable_nwbs = {}
clusterless_nwbs = {}
stable_clusterless_nwbs = {}
for subject_id in subject_ids:
    stable_nwbs[subject_id] = list( (Session & {'session_description LIKE "Spatial bandit task (regular)"'}
                                             & {"subject_id": subject_id}).fetch('nwb_file_name') )
    clusterless_nwbs[subject_id] = list(np.unique((ClusterlessAcausalResultsSummary()
                                                   & spatial_bandit_query_by_rat(rat_list=[subject_id])).fetch('nwb_file_name')))
    if subject_id == 'j16':
        stable_nwbs['j16'].remove('mediumnwb20230802_.nwb')
    if subject_id == 'chimi':
        stable_nwbs['chimi'].remove('chimi20200216_new_.nwb')
    if subject_id == 'senor':
        stable_nwbs['senor'].remove('senor20201030_.nwb')

    stable_clusterless_nwbs[subject_id] = [nwb for nwb in clusterless_nwbs[subject_id] if nwb in stable_nwbs[subject_id]]

# print(f'nwbs: {stable_clusterless_nwbs}')

is_mapped_seg_a_leaf_map = {0:False, 1:True, 2:True, 3:False, 4:True, 5:True, 6:False, 7:True, 8:True}
segs_to_patch_map = {0:1, 1:1, 2:1, 3:2, 4:2, 5:2, 6:3, 7:3, 8:3}

# get to relevant stable data only
all_rat_big_dfs_stable = {}
for subject_id in subject_ids:
    df = big_dfs[subject_id]
    df_stable = df[df['nwb_file_name'].isin(stable_clusterless_nwbs[subject_id])]
    # add some stuff for segment related analyses 
    df_stable['is_actual_seg_mapped_a_leaf'] = df_stable[['actual_segment_mapped']].applymap(is_mapped_seg_a_leaf_map.get)
    df_stable['is_mental_seg_mapped_a_leaf'] = df_stable[['mental_segment_mapped']].applymap(is_mapped_seg_a_leaf_map.get)
    df_stable['mental_patch_mapped'] = df_stable[['mental_segment_mapped']].applymap(segs_to_patch_map.get)
    all_rat_big_dfs_stable[subject_id] = df_stable

### Set up figure plotting

In [ ]:
# set up figure params
set_figure_defaults()

save_fig = True

subject_id='j16'
nonlocal_cmap = 'custom'
default_cmap = 'PuBu'
peri_nonlocal_time=.3
shading_named_color = 'cornflowerblue'#'lightsteelblue'#'royalblue'
arrow_color = 'dimgrey'
use_manual=True
extra_hpd=False
show_cbar_ticks=False

#from prior csv event detection
min_nonlocal_duration_s = 0.02
between_bin_buffer_s = 0.004

big_fig_path = fig_dir('figs26')

### Plot ex decodes

In [ ]:
#all exs
ex_datasets = {
    'j1620210707_.nwb': {
        2:  [(55, False, 0), (41, False, 0), (18, True, 1), (12, True, 0)], # (28, False, 3)
        4:  [(103, True, 1), (4, False, 0),],
        6:  [(172, False, 4)],  #(179,False,1)
        10: [(82, False, 2)],
        12: [(109, False, 1)],
    },
    'j1620210706_.nwb': {
        4: [(8, True, 0),], # (97, True, 1)],
        6: [(74, True, 0)],
    },
    'j1620210708_.nwb': {
        2: [(37, True, 1)],
    },
}

for nwb_file_name, epochs in ex_datasets.items():
    for epoch, trial_seg_event_tuples in epochs.items():

        # grab relevant data from one epoch
        acausal_results_summary, position_df, linear_position_df, mua, time_slices, subject_epoch_data = load_decode_ex_data(subject_id,nwb_file_name, epoch, all_rat_big_dfs_stable)


        # check for file path for this day epoch's figs
        day_ep_fig_path = big_fig_path + f'/' #'{nwb_file_name[0:-5]}_{epoch}/'
        if not os.path.exists(day_ep_fig_path):
            os.makedirs(day_ep_fig_path)
        fig_path = day_ep_fig_path
        print(fig_path)

        # path for example snippet exploring csv
        if np.logical_and(nwb_file_name == "j1620210707_.nwb", epoch in [4,10,12]):
            csv_path2 = f'{DATA_DIR}/ex_events_july/'
            csv_name2 = f'{nwb_file_name[:-5]}_{epoch}_binbuff{between_bin_buffer_s}_mindur{min_nonlocal_duration_s}.csv'
            csv_path = csv_path2+csv_name2
        else:
            csv_path = f'{DATA_DIR}/ex_events/{nwb_file_name[:-5]}_{epoch}.csv'
        events_per_epoch_df = pd.read_csv(csv_path)

    # plot each of those ex events
        for trial,is_first_seg_of_trial,event_num_in_trial in trial_seg_event_tuples:
            print(f"\n\n starting {nwb_file_name} epoch {epoch} trial {trial} is firstseg {is_first_seg_of_trial} event num in trial {event_num_in_trial}\n\n")

            # csv override for a few examples (else use this epoch's csv above)
            this_df = events_per_epoch_df
            csv_override = (nwb_file_name, epoch, trial, is_first_seg_of_trial, event_num_in_trial)
            if csv_override == ("j1620210707_.nwb", 4, 103, True, 1):
                # pull this one from ex_events instead of above rule
                this_df = pd.read_csv(f'{DATA_DIR}/ex_events/{nwb_file_name[:-5]}_{epoch}.csv')
            elif csv_override in [("j1620210707_.nwb", 2, 18, True, 1),
                                  ("j1620210706_.nwb", 6, 74, True, 0)]:
                #  pull these from ex_events_july
                this_df = pd.read_csv(f'{DATA_DIR}/ex_events_july/{nwb_file_name[:-5]}_{epoch}_binbuff{between_bin_buffer_s}_mindur{min_nonlocal_duration_s}.csv')

            # make dict for event from csv
            exceptions_dict = {}
            try:
                event_row = this_df[(this_df['trial'] == trial) &
                                (this_df['event_num_in_trial'] == event_num_in_trial) &
                                (this_df['is_first_seg_of_trial'] == is_first_seg_of_trial)]
                print(f"found event row for {nwb_file_name}, {trial}")

                # get only first dict
                event_dict = event_row.to_dict(orient='records').pop()

                # manual start/stop for a few examples
                if csv_override == ("j1620210706_.nwb", 6, 74, True, 0):
                    event_dict["event_start_t_manual"] = 1625602785.5143785
                    event_dict["event_stop_t_manual"] = 1625602785.6733782
                elif csv_override == ("j1620210707_.nwb", 2, 18, True, 1):
                    event_dict["event_start_t_manual"] = 1625675395.827561
                    event_dict["event_stop_t_manual"] = 1625675395.877561

                #plot with mua on top
                plot_event_v3_1_mua_ahbeh_speed(event_dict,
                    subject_epoch_data,
                    linear_position_df,
                    position_df,
                    acausal_results_summary,
                    fig_path=fig_path,
                    mua=mua,
                    save_fig=save_fig,
                    nonlocal_cmap = nonlocal_cmap, 
                    default_cmap=default_cmap, 
                    shading_named_color = shading_named_color,
                    arrow_color = arrow_color,
                    peri_nonlocal_time=peri_nonlocal_time, 
                    use_manual=use_manual, 
                    extra_hpd=extra_hpd, 
                    show_cbar_ticks=show_cbar_ticks,
                    min_nonlocal_duration_s = min_nonlocal_duration_s,
                    between_bin_buffer_s = between_bin_buffer_s)
            except Exception as e:
                print(f"exception: {e}")
                exceptions_dict[nwb_file_name][epoch][trial] = e